In [1]:
from netCDF4 import Dataset
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime,date
import plotly.express as px
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os

In [2]:
Para_original = xr.open_dataset("/home/elisacw/Documents/Modelling/para_data_ctsm_mimicsplus/clm_parameterfiles_mimicsplus_sulman/ctsm60_params.c241017.nc")
Para_original.to_netcdf("parameterfile.nc")
Para = xr.open_dataset("parameterfile.nc")
Para


<xarray.Dataset>
Dimensions:                           (pft: 79, ndecomp_pools_max: 8, segment: 4, variants: 2, ntill_stages_max: 3, ntill_intensities_max: 2)
Coordinates:
    pftname                           (pft) |S40 ...
  * segment                           (segment) |S40 b'sunlit                ...
    variantnames                      (variants) object ...
Dimensions without coordinates: pft, ndecomp_pools_max, variants, ntill_stages_max, ntill_intensities_max
Data variables: (12/427)
    C2_liq_Brun89                     float64 ...
    FUN_fracfixers                    (pft) float64 ...
    a_coef                            float64 ...
    a_exp                             float64 ...
    a_fix                             (pft) float64 ...
    accum_factor                      float64 ...
    ...                                ...
    z0v_cw                            (pft) float64 ...
    zglc                              float64 ...
    bgc_till_decompk_multipliers      (ntill_stages_max, ndecomp_pools_max, ntill_intensities_max) float64 ...
    mimics_till_decompk_multipliers   (ntill_stages_max, ndecomp_pools_max, ntill_intensities_max) float64 ...
    zbedrock                          float64 ...
    zbedrock_sf                       float64 ...
Attributes:
    Conventions:               CF-1.0
    title:                     Vegetation (Plant Function Type or PFT) constants
    NCO:                       netCDF Operators version 5.1.9 (Homepage = htt...
    nco_openmp_thread_number:  1
    history:                   Thu Feb  8 14:13:42 2024: ncatted -a history,g...

### Parameters related to MIMICS

In [3]:
# Direct litterflux from litterpools to soil pools
data_mimics_fi = np.array([0.005, 0.3])
litter_pools = ['LITm', 'LITs']
new_attrs = {'long_name': 'litter flux from litter pool to soil', 'units': '?'}
mimics_fi = xr.DataArray(
    data=data_mimics_fi,
    coords={'litter_pools': litter_pools},
    dims=['litter_pools'],
    attrs=new_attrs
)
Para['mimics_fi'] = mimics_fi

Para['mimics_fi']


<xarray.DataArray 'mimics_fi' (litter_pools: 2)>
array([0.005, 0.3  ])
Coordinates:
  * litter_pools  (litter_pools) <U4 'LITm' 'LITs'
Attributes:
    long_name:  litter flux from litter pool to soil
    units:      ?

### Parameters related to Sulman et al. 2019

When using Sulman et al. to couple MIMICS+ to vegetation, these are the parameters we will need.

List of all parameters, stating weathe rthey already exist in MIMICS+:

Parameter
Description
Value
Source / Comment



In [4]:
# Initial carbon stocks for symbiont pools
data_sulman_initial_C_stocks = np.array([100.0, 100.0, 100.0]) # fixers, miners, scavengers
new_attrs = {'long_name': 'Initial C stocks for symbionts', 'units': 'gC/m2'}
symbiont_type = ['fixer', 'scavenger', 'miner']
sulman_initial_C_stocks = xr.DataArray(
    data=data_sulman_initial_C_stocks,
    coords={'symbiont_type': symbiont_type},
    dims=['symbiont_type'],
    attrs=new_attrs
)
Para['sulman_initial_C_stocks'] = sulman_initial_C_stocks

Para['sulman_initial_C_stocks']


<xarray.DataArray 'sulman_initial_C_stocks' (symbiont_type: 3)>
array([100., 100., 100.])
Coordinates:
  * symbiont_type  (symbiont_type) <U9 'fixer' 'scavenger' 'miner'
Attributes:
    long_name:  Initial C stocks for symbionts
    units:      gC/m2

In [6]:
# Maximum symbiont growth rate
# Selected to achieve reasonable mycorrhizal biomass
new_attrs = {'long_name': 'Maximum symbiont growth rate', 'units': 'gC/m2/s'}
sulman_max_symb_growth= xr.DataArray(
    0.0000317,
    attrs=new_attrs
)
sulman_max_symb_growth
Para['sulman_max_symb_growth'] = sulman_max_symb_growth

In [7]:
# Half-saturation of intermediate C pool for symbiotic growth
# Based on typical simulated range of intermediate pool
new_attrs = {'long_name': 'Half-saturation of intermediate C pool for symbiotic growth', 'units': 'gC/m2'}
sulman_kgrowth= xr.DataArray(
    1.0,
    attrs=new_attrs
)
sulman_kgrowth
Para['sulman_kgrowth'] = sulman_kgrowth

In [8]:
# Symbiont CUE 
data_symbiont_CUE = np.array([0.5,  0.8,  0.8])
new_attrs = {'long_name': 'Symbiont growth efficiency / CUE', 'units': 'unitless'}
symbiont_type = ['fixer', 'scavenger', 'miner']

symbiont_CUE = xr.DataArray(
    data= data_symbiont_CUE,
    coords={'symbiont_type': symbiont_type},
    dims=['symbiont_type'],
    attrs=new_attrs
)
symbiont_CUE
Para['symbiont_CUE'] = symbiont_CUE
Para['symbiont_CUE']

<xarray.DataArray 'symbiont_CUE' (symbiont_type: 3)>
array([0.5, 0.8, 0.8])
Coordinates:
  * symbiont_type  (symbiont_type) <U9 'fixer' 'scavenger' 'miner'
Attributes:
    long_name:  Symbiont growth efficiency / CUE
    units:      unitless

In [9]:
# Symbiont turnover into soil organic matter pools
data_symbiont_tau = np.array([0.00000317,  0.00000317,  0.00000317])
new_attrs = {'long_name': 'Turnover of symbionts', 'units': 's-1'}
symbiont_type = ['fixer', 'scavenger', 'miner']

symbiont_tau = xr.DataArray(
    data= data_symbiont_tau,
    coords={'symbiont_type': symbiont_type},
    dims=['symbiont_type'],
    attrs=new_attrs
)
symbiont_tau
Para['symbiont_tau'] = symbiont_tau
Para['symbiont_tau']


# Fraction of symbiotic biomass turnover into SOM as necromass
# Assumed to be slightly lower than Et
new_attrs = {'long_name': 'Fraction of symbiotic biomass turnover into SOM as necromass', 'units': 'fraction'}
symbiont_necromass= xr.DataArray(
    0.7,
    attrs=new_attrs
)
symbiont_necromass
Para['symbiont_necromass'] = symbiont_necromass


# Fraction of symbiotic biomass turnover used for maintenance respiration
# Assumed to be slightly lower than Et
new_attrs = {'long_name': 'Fraction of symbiotic biomass turnover not used for maintenance respiration', 'units': 'fraction'}
symbiont_mr= xr.DataArray(
    0.3,
    attrs=new_attrs
)
symbiont_mr
Para['symbiont_mr'] = symbiont_mr

In [10]:
# Fraction symbiont necromass into soil organic matter pools
data_symbiont_tau_som = np.array([0.4, 0.2, 0.4]) # SOMc, SOMa, SOMp
new_attrs = {'long_name': 'Fraction of necromass into SOM pools', 'units': 'unitless'}
SOMpools = ['SOMc', 'SOMa', 'SOMp']

symbiont_tau_som = xr.DataArray(
    data= data_symbiont_tau_som,
    coords={'SOMpools': SOMpools},
    dims=['SOMpools'],
    attrs=new_attrs
)
symbiont_tau_som
Para['symbiont_tau_som'] = symbiont_tau_som
Para['symbiont_tau_som']


<xarray.DataArray 'symbiont_tau_som' (SOMpools: 3)>
array([0.4, 0.2, 0.4])
Coordinates:
  * SOMpools  (SOMpools) <U4 'SOMc' 'SOMa' 'SOMp'
Attributes:
    long_name:  Fraction of necromass into SOM pools
    units:      unitless

In [11]:
# C:N ratio for symbionts
data_sulman_cn_symbionts = np.array([10.0, 10.0, 10.0]) # fixers, miners, scavengers
new_attrs = {'long_name': 'C:N ratio of symbionts', 'units': 'unitless'}
symbiont_type = ['fixer', 'scavenger', 'miner']

sulman_cn_symbionts = xr.DataArray(
    data= data_sulman_cn_symbionts,
    coords={'symbiont_type': symbiont_type},
    dims=['symbiont_type'],
    attrs=new_attrs
)
sulman_cn_symbionts
Para['sulman_cn_symbionts'] = sulman_cn_symbionts
Para['sulman_cn_symbionts']

<xarray.DataArray 'sulman_cn_symbionts' (symbiont_type: 3)>
array([10., 10., 10.])
Coordinates:
  * symbiont_type  (symbiont_type) <U9 'fixer' 'scavenger' 'miner'
Attributes:
    long_name:  C:N ratio of symbionts
    units:      unitless

In [12]:
# Carbon use efficiency of mycorrhizal mining (fraction)
# Assumed to be low since mycorrhizae receive C subsidy from plants
new_attrs = {'long_name': 'Carbon use efficiency of mycorrhizal mining', 'units': 'fraction'}
sulman_cue_mine= xr.DataArray(
    0.05,
    attrs=new_attrs
)
sulman_cue_mine
Para['sulman_cue_mine'] = sulman_cue_mine

# Nitrogen use efficiency of mycorrhizal mining (fraction)
# Assumed to be high
new_attrs = {'long_name': 'Nitrogen use efficiency of mycorrhizal mining', 'units': 'fraction'}
sulman_nue_mine= xr.DataArray(
    0.9,
    attrs=new_attrs
)
sulman_nue_mine
Para['sulman_nue_mine'] = sulman_nue_mine

In [13]:
# Maximum root active nitrate uptake rate (kg N m-3 s-1)
# Fit to observed N uptake rates in AM systems, assuming max root uptake rate is lower than that of mycorrhizae
new_attrs = {'long_name': 'Maximum root active nitrate uptake rate', 'units': 'gN/m3/s'}
sulman_root_no3= xr.DataArray(
    0.00000317,
    attrs=new_attrs
)
sulman_root_no3
Para['sulman_root_no3'] = sulman_root_no3

# Maximum root active ammonium uptake rate (kg N m-3 s-1)
# Fit to observed N uptake rates in AM systems, assuming max root uptake rate is lower than that of mycorrhizae
new_attrs = {'long_name': 'Maximum root active ammonium uptake rate', 'units': 'gN/m3/s'}
sulman_root_nh4= xr.DataArray(
    0.00000317,
    attrs=new_attrs
)
sulman_root_nh4
Para['sulman_root_nh4'] = sulman_root_nh4

In [14]:
# Maximum NH4+ immobilization rate
# Assumes inorganic N pools can be assimilated by microbes at a one day time scale
new_attrs = {'long_name': 'Maximum NH4+ immobilization rate', 'units': 's-1'}
sulman_v_nh4= xr.DataArray(
    0.00422,
    attrs=new_attrs
)
sulman_v_nh4
Para['sulman_v_nh4'] = sulman_v_nh4


# Maximum NO3- immobilization rate
# Assumes inorganic N pools can be assimilated by microbes at a one day time scale
new_attrs = {'long_name': 'Maximum NO3- immobilization rate', 'units': 's-1'}
sulman_v_no3= xr.DataArray(
    0.00422,
    attrs=new_attrs
)
sulman_v_no3
Para['sulman_v_no3'] = sulman_v_no3

In [15]:
# Half-saturation nitrate concentration for root active uptake (kg N m-3) 
# Based on typical range of simulated inorganic N concentrationvalues
new_attrs = {'long_name': 'Half-saturation nitrate concentration for root active uptake', 'units': 'gN/m3'}
sulman_km_no3= xr.DataArray(
   300, #0.08, # 5.0,
    attrs=new_attrs
)
sulman_km_no3
Para['sulman_km_no3'] = sulman_km_no3

# Half-saturation ammonium concentration for root active uptake
# Based on typical range of simulated inorganic N concentrationvalues
new_attrs = {'long_name': 'Half-saturation nitrate concentration for root active uptake', 'units': 'gN/m3'}
sulman_km_nh4= xr.DataArray(
    300, #0.08, # 5.0,
    attrs=new_attrs
)
sulman_km_nh4
Para['sulman_km_nh4'] = sulman_km_nh4

In [16]:
# Half-saturation inorganic N concentration for mycorrhizal uptake
# Based on typical range of simulated inorganic N concentration values
new_attrs = {'long_name': 'Half-saturation inorganic N concentration for mycorrhizal uptake', 'units': 'gN/m3'}
sulman_k_scav_Ninorg= xr.DataArray(
    1.0,
    attrs=new_attrs
)
sulman_k_scav_Ninorg
Para['sulman_k_scav_Ninorg'] = sulman_k_scav_Ninorg

# Maximum N uptake rate by scavenging mycorrhizae
# Fit to observed N acquisition in AM systems
new_attrs = {'long_name': 'Maximum N uptake rate by scavenging mycorrhizae', 'units': 'gN/m3/s'}
sulman_v_scav= xr.DataArray(
   0.00000317,
    attrs=new_attrs
)
sulman_v_scav
Para['sulman_v_scav'] = sulman_v_scav

In [17]:
# PROBABLY DO NOT USE
# Half-saturation mycorrhizal biomass concentration for scavenging
# Based on typical mycorrhizal biomass of 0.3 kg m-3 (Zhu and Miller, 2003)
new_attrs = {'long_name': 'Half-saturation mycorrhizal biomass concentration for scavenging', 'units': 'gC/m3'}
sulman_k_scav= xr.DataArray(
    0.01,
    attrs=new_attrs
)
sulman_k_scav
Para['sulman_k_scav'] = sulman_k_scav

# Half-saturation mycorrhizal biomass for mining decomposition (g microbial biomass C g substrate C-1)
# Assumed same as that of free microbial biomass
new_attrs = {'long_name': 'Half-saturation mycorrhizal biomass concentration for mining', 'units': 'gC/m3'}
sulman_km_mine= xr.DataArray(
    0.01,
    attrs=new_attrs
)
sulman_km_mine
Para['sulman_km_mine'] = sulman_km_mine

In [18]:
# Maximum decomposition rate at reference temperature for mycorrhizal mining
# Assumed to be more efficient for slow SOM (compared to free-living microbes) and less efficient for more labile SOM
# data_sulman_vmax_ref_mine = [0.000000000285, 0.0000000000792, 0.000000000285] # (fast),(slow),(necromass)
#data_sulman_vmax_ref_mine = [0.000143, 0.000000793, 0.000019 ] # (fast),(slow),(necromass)
#new_attrs = {'long_name': 'Maximum decomposition rate at reference temperature for mycorrhizal mining', 'units': 's-1'}
#sulman_vmax_ref_mine= xr.DataArray(
# data= data_sulman_vmax_ref_mine,
#    coords={'variant': [0, 1, 2]},
#    dims=['variant'],
#    attrs=new_attrs
#)
#sulman_vmax_ref_mine
#Para['sulman_vmax_ref_mine'] = sulman_vmax_ref_mine
#Para['sulman_vmax_ref_mine']

new_attrs = {'long_name': 'Maximum decomposition rate at reference temperature for mycorrhizal mining', 'units': 's-1'}
sulman_vmax_ref_mine= xr.DataArray(
    0.000000793,
    attrs=new_attrs
)
sulman_vmax_ref_mine
Para['sulman_vmax_ref_mine'] = sulman_vmax_ref_mine

In [19]:
# N fixation rate per unit symbiotic biomass
# Estimated from nodule mass andN fixation rates of Batterman etal.(2013)
new_attrs = {'long_name': 'N fixation rate per unit symbiotic biomass', 'units': 'gN/gC/s'}
sulman_rfix= xr.DataArray(
    0.00000317,
    attrs=new_attrs
)
sulman_rfix
Para['sulman_rfix'] = sulman_rfix

In [20]:
# Turnover time of intermediate C pool
# Assumed to have one day residence time
new_attrs = {'long_name': 'Turnover time of intermediate C pool', 'units': 's-1'}
sulman_tau_int= xr.DataArray(
    0.00422,
    attrs=new_attrs
)
sulman_tau_int
Para['sulman_tau_int'] = sulman_tau_int

In [21]:
# Radius of the rhizosphere (m)
new_attrs = {'long_name': 'Radius of the rhizosphere', 'units': 'm'}
sulman_r_rhiz= xr.DataArray(
    0.0001,
    attrs=new_attrs
)
sulman_r_rhiz
Para['sulman_r_rhiz'] = sulman_r_rhiz

In [22]:
# Vegetation N uptake rate from intermediate N pool
# Selected t be slightly higher than mycorrhizal growth rate
new_attrs = {'long_name': 'Vegetation N uptake rate from intermediate N pool', 'units': 's-1'}
sulman_rup_veg= xr.DataArray(
    0.00000317,
    attrs=new_attrs
)
sulman_rup_veg
Para['sulman_rup_veg'] = sulman_rup_veg

In [23]:
Para.to_netcdf('clm_params_mimicsplus_sulman_netcdf4.nc')

In [24]:
para1=xr.open_dataset("clm_params_mimicsplus_sulman_netcdf4.nc")
para1

<xarray.Dataset>
Dimensions:                           (pft: 79, ndecomp_pools_max: 8, segment: 4, variants: 2, ntill_stages_max: 3, ntill_intensities_max: 2, litter_pools: 2, symbiont_type: 3, SOMpools: 3)
Coordinates:
    pftname                           (pft) |S40 ...
  * segment                           (segment) |S40 b'sunlit                ...
    variantnames                      (variants) object ...
  * litter_pools                      (litter_pools) object 'LITm' 'LITs'
  * symbiont_type                     (symbiont_type) object 'fixer' ... 'miner'
  * SOMpools                          (SOMpools) object 'SOMc' 'SOMa' 'SOMp'
Dimensions without coordinates: pft, ndecomp_pools_max, variants, ntill_stages_max, ntill_intensities_max
Data variables: (12/455)
    C2_liq_Brun89                     float64 ...
    FUN_fracfixers                    (pft) float64 ...
    a_coef                            float64 ...
    a_exp                             float64 ...
    a_fix                             (pft) float64 ...
    accum_factor                      float64 ...
    ...                                ...
    sulman_km_mine                    float64 ...
    sulman_vmax_ref_mine              float64 ...
    sulman_rfix                       float64 ...
    sulman_tau_int                    float64 ...
    sulman_r_rhiz                     float64 ...
    sulman_rup_veg                    float64 ...
Attributes:
    Conventions:               CF-1.0
    title:                     Vegetation (Plant Function Type or PFT) constants
    NCO:                       netCDF Operators version 5.1.9 (Homepage = htt...
    nco_openmp_thread_number:  1
    history:                   Thu Feb  8 14:13:42 2024: ncatted -a history,g...